# CMSC 173 &middot; Module 13 &middot; Example 1
## Convolution, by hand

A 5 &times; 5 grid of brightness values and a 3 &times; 3 kernel that responds where brightness rises from left to right. Slide the kernel, multiply cell by cell, add: that is a convolution.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides (NumPy; a newer version can differ slightly for the trained networks);
change something and see what moves.

**One window.** The kernel lies on the top-left 3 &times; 3 window; the nine products add to $Y[0,0]$.

In [ ]:
import numpy as np

X = np.array([[1, 1, 5, 5, 5]] * 3 + [[1, 1, 1, 5, 5]] * 2)   # 5 x 5 image: dark left, bright right
K = np.array([[-1, 0, 1]] * 3)                                # 3 x 3 kernel: right minus left
print(X[0:3, 0:3] * K)                                        # the nine products
print((X[0:3, 0:3] * K).sum())                                # 12

**The whole feature map.** Every position the kernel fits: 3 across, 3 down.

In [ ]:
def conv(img, k, stride=1, pad=0):
    img = np.pad(img, pad)                                    # pad: rings of zeros around the image
    n, kk = img.shape[0], k.shape[0]
    o = (n - kk) // stride + 1                                # the output-size formula
    Y = np.zeros((o, o), dtype=int)
    for i in range(o):
        for j in range(o):
            window = img[i * stride:i * stride + kk, j * stride:j * stride + kk]
            Y[i, j] = (window * k).sum()                      # cell by cell, then add
    return Y

print(conv(X, K))             # [[12 12 0] [8 12 4] [4 12 8]]
print(conv(X, K, stride=2))   # [[12 0] [4 8]]: the four corners
print(conv(X, K, pad=1))      # 5 x 5; the last column is negative: a false edge from the zeros

**The output-size formula**, $o = \lfloor (n + 2p - k)/s \rfloor + 1$, on the three maps above and on the first layer of ResNet (a 224 &times; 224 photo, a 7 &times; 7 kernel, padding 3, stride 2).

In [ ]:
osize = lambda n, k, p, s: (n + 2 * p - k) // s + 1
print(osize(5, 3, 0, 1), osize(5, 3, 1, 1), osize(5, 3, 0, 2), osize(224, 7, 3, 2))   # 3 5 2 112

**The same kernel on a real handwritten 0** (8 &times; 8 pixels, values 0 to 16), with padding 1.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

zero = load_digits().images[0]
E = conv(zero.astype(int), K, pad=1)
print(E.max(), E.min())        # 40 -35: the left side of a stroke, and its right side
fig, ax = plt.subplots(1, 2, figsize=(8, 3.6))
ax[0].imshow(zero, cmap="Greys"); ax[0].set_title("a handwritten 0")
ax[1].imshow(E, cmap="RdBu_r", vmin=-40, vmax=40); ax[1].set_title("after the edge kernel")
plt.show()

**Try it.** Turn the kernel on its side, `np.array([[-1] * 3, [0] * 3, [1] * 3])`, so it responds where brightness rises from top to bottom. Which parts of the 0 light up now?